# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get and print high-level metadata info
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}\nVersion: {getattr(metadata, 'version', 'N/A')}\nPublished: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This information is essential for loading and referencing specific portions of the dataset.

- Each **RecordSet** and **Field** can be accessed and referenced by its unique `@id`.
- In this step, we will list all record sets, including their names and `@id`s, and enumerate their fields as well.


In [ ]:
# List all record sets, their IDs, and fields' IDs
record_sets = dataset.record_sets
print(f"Total RecordSets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {getattr(rs, 'name', 'N/A')} | @id: {rs.id}")
    fields = rs.fields
    print("  Fields (by @id and name):")
    for f in fields:
        print(f"    - {f.id} : {f.name} (type: {getattr(f, 'dataType', 'N/A')})")
    print('-' * 60)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

- You can select a record set by its `@id` from the output above. As an example, we'll extract all record sets here. 
- Data for each record set is loaded as a pandas DataFrame and stored in a `dataframes` dictionary, with the key being the record set `@id`.


In [ ]:
# Load all record sets into DataFrames, indexed by their @id
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for rs in dataset.record_sets:
    print(f"Loading record set: {rs.name} (@id={rs.id})")
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Number of records: {len(df)}\n")

# For demonstration, pick the first record set (you can change as needed)
if len(record_set_ids) > 0:
    selected_record_set_id = record_set_ids[0]
    print(f"Selected RecordSet for demonstration: {selected_record_set_id}")
    print(dataframes[selected_record_set_id].head())
else:
    print("No record sets found in the dataset.")

# Save for later
df = dataframes[selected_record_set_id] if len(record_set_ids) > 0 else pd.DataFrame()


## 4. Exploratory Data Analysis (EDA)
Let's do some basic data processing:
- Select a numeric field by its `@id` (shown in previous steps)
- Filter records based on a threshold value for this field
- Normalize the numeric field (zero mean, unit variance)
- Optionally, group by another field and report means

_Pick field IDs and types from the earlier cell's printout._

In [ ]:
# Choose the record set and its numeric field by @id
# Substitute your own choices below as needed, based on printed overview

# Example defaults:
record_set_id = selected_record_set_id  # from previous cell
df = dataframes[record_set_id]

# For illustration, pick the first numeric column
numeric_candidates = df.select_dtypes(include='number').columns.tolist()
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]  # e.g. '@id' of a numeric field like coverage_percentage, molecular_weight, etc.
    print(f"Using numeric field: {numeric_field_id}")
else:
    print("No numeric fields found for EDA.")
    numeric_field_id = None

if numeric_field_id:
    # Filter records where value > threshold
    threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records: {len(filtered_df)}/{len(df)} rows with {numeric_field_id} > {threshold:.2f}")
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to pick an appropriate group field (categorical, not unique, e.g. sample id)
    candidate_group_fields = [c for c in df.columns if df[c].dtype == 'O' and df[c].nunique() < len(df)/2 and c != numeric_field_id]
    group_field_id = candidate_group_fields[0] if candidate_group_fields else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
    else:
        print("No suitable group field found; skipping groupby.")
else:
    filtered_df = df
    group_field_id = None


## 5. Visualization
Visualize the distribution of the numeric field(s) and, if applicable, compare group means.

- Histogram for the selected numeric field
- Bar plot for group means (if grouped)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and not filtered_df.empty:
    plt.figure(figsize=(6, 4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

if numeric_field_id and group_field_id and 'grouped_df' in locals():
    plt.figure(figsize=(8, 4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
    plt.title(f"Mean {numeric_field_id} grouped by {group_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
elif numeric_field_id:
    print("No grouping field available for a grouped bar plot.")

## 6. Conclusion
- We demonstrated how to load a FAIR^2 dataset via its Croissant schema using `mlcroissant`, exploring available record sets, fields, and data.
- Basic EDA such as filtering, normalizing, and grouping by field was performed, using the `@id` reference convention throughout.
- The above workflow can be adapted for any other Croissant-compliant dataset.

**Next Steps:**
- Explore additional record sets or fields as needed
- Try more advanced data processing or machine learning workflows